In [7]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from pathlib import Path
from config import get_rng, count_missing, SAMPLE_NUM, MISSING_DATA_LEVELS

rng = get_rng()

PROCESSED_DATA_PATH = "processed_datasets"

def introduce_MCAR(df, missing_fraction, rng):
    df_missing = df.copy()
    df_missing = df_missing[df_missing.columns[:-1]]

    total_cells = df_missing.size
    n_missing = int(total_cells * missing_fraction)

    flat_indices = rng.choice(total_cells, n_missing, replace=False)

    rows = flat_indices // df_missing.shape[1]
    cols = flat_indices % df_missing.shape[1]

    for r, c in zip(rows, cols):
        df_missing.iat[r, c] = np.nan

    df_missing = pd.concat([df_missing, df[df.columns[-1]]], axis=1)

    return df_missing

def introduce_MAR(df, missing_fraction, rng):
    df_missing = df.copy()
    
    features = df_missing.iloc[:, :-1].copy()
    n_rows, n_cols = features.shape
    missing_mask = np.zeros((n_rows, n_cols), dtype=bool)

    for c in range(1, n_cols):
        parent_values = features.iloc[:, c - 1].values
        
        std_val = np.std(parent_values)
        if std_val == 0:
            z_scores = np.zeros(n_rows)
        else:
            z_scores = (parent_values - np.mean(parent_values)) / std_val

        raw_probs = stats.norm.cdf(z_scores)

        if raw_probs.mean() > 0:
            scaled_probs = raw_probs * (missing_fraction / raw_probs.mean())
        else:
            scaled_probs = np.full(n_rows, missing_fraction)

        scaled_probs = np.clip(scaled_probs, 0.01, 0.99)
        
        missing_mask[:, c] = rng.random(n_rows) < scaled_probs

    current_missing = missing_mask.sum()
    target_missing = int(n_rows * n_cols * missing_fraction)
    needed_missing = target_missing - current_missing

    if needed_missing > 0:
        available_indices = np.argwhere(~missing_mask)
        
        if len(available_indices) >= needed_missing:
            chosen_coords = rng.choice(len(available_indices), size=needed_missing, replace=False)
            for idx in chosen_coords:
                r, c = available_indices[idx]
                missing_mask[r, c] = True

    features[missing_mask] = np.nan

    res_df = pd.concat([features, df_missing.iloc[:, [-1]]], axis=1)

    return res_df

def introduce_MNAR(df, missing_fraction, rng):
    df_missing = df.copy()
    
    features = df_missing.iloc[:, :-1].copy()
    n_rows, n_cols = features.shape
    missing_mask = np.zeros((n_rows, n_cols), dtype=bool)

    for c in range(n_cols):
        col_values = features.iloc[:, c].values
        
        std_val = np.std(col_values)
        if std_val == 0:
            z_scores = np.zeros(n_rows)
        else:
            z_scores = (col_values - np.mean(col_values)) / std_val

        raw_probs = stats.norm.cdf(z_scores)

        if raw_probs.mean() > 0:
            scaled_probs = raw_probs * (missing_fraction / raw_probs.mean())
        else:
            scaled_probs = np.full(n_rows, missing_fraction)

        scaled_probs = np.clip(scaled_probs, 0.01, 0.99)
        
        missing_mask[:, c] = rng.random(n_rows) < scaled_probs

    current_missing = missing_mask.sum()
    target_missing = int(n_rows * n_cols * missing_fraction)
    needed_missing = target_missing - current_missing

    if needed_missing > 0:
        available_indices = np.argwhere(~missing_mask)
        if len(available_indices) >= needed_missing:
            chosen_coords = rng.choice(len(available_indices), size=needed_missing, replace=False)
            for idx in chosen_coords:
                r, c = available_indices[idx]
                missing_mask[r, c] = True

    features[missing_mask] = np.nan

    res_df = pd.concat([features, df_missing.iloc[:, [-1]]], axis=1)

    return res_df

In [8]:
# Loading data

def load_data_from_folder(folder_path):
    folder = Path(folder_path)
    df_list = []
    file_list = []
    for file in folder.glob("*.csv"):
        df_list.append(pd.read_csv(file))
        file_list.append(file)
    return file_list, df_list


file_list, df_list = load_data_from_folder(PROCESSED_DATA_PATH)

***Introducing syntetic missing values nad splitting into train and test sets***

In [ ]:
seeds = rng.integers(low=0, high=10000, size=SAMPLE_NUM)

for i, seed in enumerate(seeds):
    print(f"\n--- Sample {i+1} with seed {seed} ---")
    for file in file_list:
        df = pd.read_csv(file)
        for lvl in MISSING_DATA_LEVELS:
            train_df, test_df = train_test_split(df, test_size=0.2, random_state=seed, stratify=df[df.columns[-1]])
            train_df_mcar = introduce_MCAR(train_df, lvl, rng)
            train_df_mar = introduce_MAR(train_df, lvl, rng)
            train_df_mnar = introduce_MNAR(train_df, lvl, rng)
            total, missing_mcar, percent_mcar = count_missing(train_df_mcar)
            total, missing_mar, percent_mar = count_missing(train_df_mar)
            total, missing_mnar, percent_mnar = count_missing(train_df_mnar)

            print(f"\nTotal entries in train split: {total}")
            print(f"--- Missing percentage: {int(lvl*100)}% ---")
            print(f"Missing values after MCAR modification: {missing_mcar} ({percent_mcar:.2f}%)")
            print(f"Missing values after MAR modification: {missing_mar} ({percent_mar:.2f}%)")
            print(f"Missing values after MNAR modification: {missing_mnar} ({percent_mnar:.2f}%)")

            train_output_dir = Path(f"missing_train_splits/sample{i+1}")
            train_output_dir.mkdir(parents=True, exist_ok=True)
            train_output_mcar = train_output_dir / f"{file.stem}_train_mcar{int(lvl*100)}.csv"
            train_output_mar = train_output_dir / f"{file.stem}_train_mar{int(lvl*100)}.csv"
            train_output_mnar = train_output_dir / f"{file.stem}_train_mnar{int(lvl*100)}.csv"

            train_df_mcar.to_csv(train_output_mcar, index=False)
            print(f"Saved: {train_output_mcar}")
                        
            train_df_mar.to_csv(train_output_mar, index=False)
            print(f"Saved: {train_output_mar}")

            train_df_mnar.to_csv(train_output_mnar, index=False)
            print(f"Saved: {train_output_mnar}")

            test_output_dir = Path(f"missing_test_splits/sample{i+1}")
            test_output_dir.mkdir(parents=True, exist_ok=True)
            test_output_name = test_output_dir / f"{file.stem}_test_miss{int(lvl*100)}.csv"
            test_df.to_csv(test_output_name, index=False)
            print(f"Saved: {test_output_name}")


--- Sample 1 with seed 577 ---

Total entries in train split: 504000
--- Missing percentage: 10% ---
Missing values after MCAR modification: 50400 (10.00%)
Missing values after MAR modification: 50400 (10.00%)
Missing values after MNAR modification: 50400 (10.00%)
Saved: missing_train_splits\sample1\diabetes_train_mcar10.csv
Saved: missing_train_splits\sample1\diabetes_train_mar10.csv
Saved: missing_train_splits\sample1\diabetes_train_mnar10.csv
Saved: missing_test_splits\sample1\diabetes_test_miss10.csv

Total entries in train split: 504000
--- Missing percentage: 20% ---
Missing values after MCAR modification: 100800 (20.00%)
Missing values after MAR modification: 100800 (20.00%)
Missing values after MNAR modification: 100800 (20.00%)
Saved: missing_train_splits\sample1\diabetes_train_mcar20.csv
Saved: missing_train_splits\sample1\diabetes_train_mar20.csv
Saved: missing_train_splits\sample1\diabetes_train_mnar20.csv
Saved: missing_test_splits\sample1\diabetes_test_miss20.csv

Total 